In [ ]:
%pip install Pillow
%pip install --upgrade google-genai fsspec gcsfs google-auth google-api-core
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [1]:
import google.auth
_, PROJECT_ID = google.auth.default()

In [2]:
from google.genai import types
from google import genai
import concurrent.futures
from PIL import Image
from collections.abc import Callable

def process_image_embedding_worker(id: str, file_path: str):
    client = genai.Client(vertexai=True, location="global", project=PROJECT_ID)
    try:            
        # 임베딩 API 호출
        result = client.models.embed_content(
            model='gemini-embedding-2',
            contents=[
                types.Part.from_uri(file_uri=file_path),
            ],
            config = types.EmbedContentConfig(
                output_dimensionality=768,
                http_options=types.HttpOptions(
                    retry_options=types.HttpRetryOptions(
                        attempts=100,
                        initial_delay=1.0,
                        max_delay=5.0
                    )
                )
            )
        )        
        return [id, file_path, result.embeddings[0].values]
    except Exception as e:
        print(f"API 호출 또는 처리 중 오류 발생 ({id}): {e}")
        return None

def process_text_content_embedding_worker(id: str, content: str):
    client = genai.Client(vertexai=True, location="global", project=PROJECT_ID)
    try:            
        # 임베딩 API 호출
        result = client.models.embed_content(
            model='gemini-embedding-2',
            contents=[
                types.Part.from_text(text=f"title: none | text: {content}"),
            ],
            config = types.EmbedContentConfig(
                output_dimensionality=768,
                http_options=types.HttpOptions(
                    retry_options=types.HttpRetryOptions(
                        attempts=100,
                        initial_delay=1.0,
                        max_delay=5.0
                    )
                )
            )
        )        
        return [id, content, result.embeddings[0].values]
    except Exception as e:
        print(f"API 호출 또는 처리 중 오류 발생 ({id}): {e}")
        return None

def process_image_thumbnail_worker(id: str, file_path: str):
    output = f"./thumbnails/{id}.webp"
    with Image.open(file_path) as img:
        # RGB 모드로 변환 (PNG나 특정 JPG의 투명도/색상 프로필 오류 방지)
        if img.mode in ("RGBA", "P"):
            img = img.convert("RGB")        
        # 한 변의 최대 해상도를 400px로 제한 (비율 유지)
        img.thumbnail((400, 400))        
        # WebP 포맷으로 저장 (quality는 0~100 사이, 80 정도면 화질 저하 없이 용량이 대폭 줄어듭니다)
        img.save(output, format="WEBP", quality=80)
    return [id, file_path, output]

def process_parallel(tasks: list, processor: Callable[[str], str], max_workers: int = 4):
    """
    여러개의 Task를 ThreadPoolExecutor를 이용해 병렬로 동작합니다.
    """
    results_with_index = []
    
    print(f"총 {len(tasks)}개의 데이터를 {max_workers}개의 스레드로 처리합니다...")
    
    # 스레드 풀을 이용한 병렬 처리
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        # 입력된 Task 리스트의 인덱스를 유지하여 순서 보장
        future_to_index = {
            executor.submit(processor, id, data): idx
            for idx, (id, data) in enumerate(tasks)
        }
        
        # 완료되는 순서대로 결과 수집
        i = 0
        for future in concurrent.futures.as_completed(future_to_index):
            i += 1
            if i % 1000 == 0:
                print(f"{i} 개 처리 완료")
            idx = future_to_index[future]
            try:
                res = future.result()
                if res:
                    results_with_index.append((idx, res))
            except Exception as e:
                print(f"스레드 실행 중 예외 발생 (Index {idx}): {e}")

    # 병렬 처리로 인해 뒤섞인 결과를 원래 입력 순서(index)대로 정렬
    results_with_index.sort(key=lambda x: x[0])
    
    # 최종 결과 데이터만 리스트로 반환
    final_results = [res[1] for res in results_with_index]
    return final_results

In [3]:
# 이미지 임베딩 생성
from pathlib import Path
# ! 명령어를 통해 gcloud 결과를 file_list 변수에 리스트 형태로 저장
# ** 와일드카드를 사용해 하위 폴더의 모든 파일을 재귀적으로 탐색합니다.
file_list = !gcloud storage ls gs://jk-amazon-products/**

print(f"총 {len(file_list)}개의 파일을 찾았습니다.")
images_list = []

for file in file_list:
    if file.endswith(('.png', '.jpg')):
        # Convert gs:// to public gcs path
        images_list.append((Path(file).stem, "https://storage.googleapis.com/" + file[5:]))
print(f"총 {len(images_list)}개의 이미지 파일을 찾았습니다.")

총 398213개의 파일을 찾았습니다.
총 398212개의 이미지 파일을 찾았습니다.


In [4]:
image_embeddings = process_parallel(images_list, process_image_embedding_worker, 64)

총 398212개의 데이터를 64개의 스레드로 처리합니다...
1000 개 처리 완료
2000 개 처리 완료
3000 개 처리 완료
4000 개 처리 완료
5000 개 처리 완료
6000 개 처리 완료
7000 개 처리 완료
8000 개 처리 완료
9000 개 처리 완료
10000 개 처리 완료
11000 개 처리 완료
12000 개 처리 완료
13000 개 처리 완료
14000 개 처리 완료
15000 개 처리 완료
16000 개 처리 완료
17000 개 처리 완료
18000 개 처리 완료
19000 개 처리 완료
20000 개 처리 완료
21000 개 처리 완료
22000 개 처리 완료
23000 개 처리 완료
24000 개 처리 완료
25000 개 처리 완료
26000 개 처리 완료
27000 개 처리 완료
28000 개 처리 완료
29000 개 처리 완료
30000 개 처리 완료
31000 개 처리 완료
32000 개 처리 완료
33000 개 처리 완료
34000 개 처리 완료
35000 개 처리 완료
36000 개 처리 완료
37000 개 처리 완료
38000 개 처리 완료
39000 개 처리 완료
40000 개 처리 완료
41000 개 처리 완료
42000 개 처리 완료
43000 개 처리 완료
44000 개 처리 완료
45000 개 처리 완료
46000 개 처리 완료
47000 개 처리 완료
48000 개 처리 완료
49000 개 처리 완료
50000 개 처리 완료
51000 개 처리 완료
52000 개 처리 완료
53000 개 처리 완료
54000 개 처리 완료
55000 개 처리 완료
56000 개 처리 완료
57000 개 처리 완료
58000 개 처리 완료
59000 개 처리 완료
60000 개 처리 완료
61000 개 처리 완료
62000 개 처리 완료
63000 개 처리 완료
64000 개 처리 완료
65000 개 처리 완료
66000 개 처리 완료
67000 개 처리 완료
68000 개 처리 완료
69000 개 처리 완료
70000 개 

In [18]:
import pandas as pd
df_image_embeddings = pd.DataFrame([
    {"id": item[0], "vectors": {"image_embedding": item[2]}} 
    for item in image_embeddings
])
df_image_embeddings

,id,vectors
0,00000529,"{'image_embedding': [-0.03478194, -0.005159070..."
1,00003a93,"{'image_embedding': [-0.028530316, -0.03578147..."
2,000088e1,"{'image_embedding': [-0.03092741, 0.0020457155..."
3,0000b301,"{'image_embedding': [0.009060884, -0.030949578..."
4,0000b9b8,"{'image_embedding': [-0.044396702, -0.01553370..."
...,...,...
398184,ffff4040,"{'image_embedding': [-0.011574976, 0.01045859,..."
398185,ffff87d7,"{'image_embedding': [-0.047705553, -0.05032951..."
398186,ffffc378,"{'image_embedding': [0.011752412, -0.017997367..."
398187,ffffd5c6,"{'image_embedding': [-0.030390358, 0.003537298..."


In [7]:
# 전체 메타데이터 획득
all_df = []
for i in range(16):
    df = pd.read_json(f'./abo-listings/listings/metadata/listings_{i}.json', lines=True)
    df = df[['main_image_id', 'product_type', 'other_image_id', 'item_name', 'item_keywords', 'country']]
    # 1. other_image_id의 결측치(NaN)를 빈 리스트([])로 변환
    # (리스트끼리 더하기 위해 전처리가 필요합니다)
    safe_other_ids = df['other_image_id'].apply(lambda x: x if isinstance(x, list) else [])
    
    # 2. main_image_id를 리스트로 감싸고, safe_other_ids와 더해서 하나의 큰 리스트로 만듭니다.
    # 예: ['81iZlv3bjpL'] + ['91mIRxgziUL', '91eqBkW06wL'] -> ['81iZlv3bjpL', '91mIRxgziUL', '91eqBkW06wL']
    df['combined_ids'] = df['main_image_id'].apply(lambda x: [x]) + safe_other_ids
    
    # 3. 합쳐진 리스트 컬럼을 기준으로 행을 확장(explode)합니다.
    df = df.explode('combined_ids')
    
    # 4. 확장된 값들을 main_image_id 컬럼에 덮어쓰고, 불필요해진 컬럼들을 삭제합니다.
    df['main_image_id'] = df['combined_ids']
    
    # 5. item_name: 첫 번째 딕셔너리에서 'value' 값만 가져오기 (비어있으면 빈 문자열)
    df['item_name'] = df['item_name'].apply(
        lambda x: x[0].get('value', '') if isinstance(x, list) and x else ''
    )
    
    # 6. item_keywords: 배열 내 딕셔너리를 돌면서 # 붙이고 공백 없앤 뒤 쉼표로 연결
    df['item_keywords'] = df['item_keywords'].apply(
        lambda x: " ".join([f"#{i.get('value', '')}" for i in x]) if isinstance(x, list) else ''
    )
    
    # 7. country 컬럼을 '[국가코드] ' 형태로 만들어 item_name 앞에 붙이기
    df['item_name'] = "[" + df['country'] + "] " + df['item_name']

    # 8. [{'value': 'SHOES'}, {'value': 'BOOTS'}] 형태를 ['SHOES', 'BOOTS'] 형태의 깔끔한 리스트로 변환
    df['product_type'] = df['product_type'].apply(
        lambda x: [i.get('value', '') for i in x] if isinstance(x, list) else []
    )
    
    # 9. 리스트의 원소 개수만큼 행(row)을 확장 (explode)
    df = df.explode('product_type', ignore_index=True)
    
    df = df.drop(columns=['other_image_id', 'combined_ids', 'country'])
    
    # 10. 인덱스 재정렬
    df = df.reset_index(drop=True)
    all_df.append(df)
df_meta = pd.concat(all_df, ignore_index=True)
df_meta

,main_image_id,product_type,item_name,item_keywords
0,81iZlv3bjpL,SHOES,[NL] Amazon-merk - vinden. Dames Leder Geslote...,#block heel shoes #loafer shoes #loafers #meta...
1,91mIRxgziUL,SHOES,[NL] Amazon-merk - vinden. Dames Leder Geslote...,#block heel shoes #loafer shoes #loafers #meta...
2,91eqBkW06wL,SHOES,[NL] Amazon-merk - vinden. Dames Leder Geslote...,#block heel shoes #loafer shoes #loafers #meta...
3,A1BHZSKNbkL,SHOES,[NL] Amazon-merk - vinden. Dames Leder Geslote...,#block heel shoes #loafer shoes #loafers #meta...
4,619y9YG9cnL,HARDWARE,"[MX] 22"" Bottom Mount Drawer Slides, White Pow...",#22 #Pistola #Montaje bajo #diapositivas #Exte...
...,...,...,...,...
711223,81fnxlzmiYL,SHOES,[CA] Amazon Brand - find. Men’s Espadrilles,
711224,81YNklsz4dL,SHOES,[CA] Amazon Brand - find. Men’s Espadrilles,
711225,A1+VHY23fgL,SHOES,[CA] Amazon Brand - find. Men’s Espadrilles,
711226,913lKMbp7OL,SHOES,[CA] Amazon Brand - find. Men’s Espadrilles,


In [10]:
df_image_path = pd.read_csv('./abo-images-original/images/metadata/images.csv')[['image_id', 'path']]
df_image_path['path'] = df_image_path['path'].str[3:-4]
df_image_path

,image_id,path
0,010-mllS7JL,14fe8812
1,01dkn0Gyx0L,daab0cad
2,01sUPg0387L,d2daaae9
3,1168jc-5r1L,3a4e88e6
4,11RUV5Fs65L,d91ab9cf
...,...,...
398207,B1zv8OpTkBS,6d49d130
398208,B1zwflWhPIS,b163e0ea
398209,C1lf45DhhRS,a116d9d1
398210,C1pEt6jBLiS,9c3e1158


In [11]:
df_merged = pd.merge(df_meta, df_image_path, left_on='main_image_id', right_on='image_id', how='left')
df_merged = df_merged[['path', 'product_type', 'item_name', 'item_keywords']]
df_merged

,path,product_type,item_name,item_keywords
0,8ccb5859,SHOES,[NL] Amazon-merk - vinden. Dames Leder Geslote...,#block heel shoes #loafer shoes #loafers #meta...
1,13741743,SHOES,[NL] Amazon-merk - vinden. Dames Leder Geslote...,#block heel shoes #loafer shoes #loafers #meta...
2,3062eff1,SHOES,[NL] Amazon-merk - vinden. Dames Leder Geslote...,#block heel shoes #loafer shoes #loafers #meta...
3,8cf58314,SHOES,[NL] Amazon-merk - vinden. Dames Leder Geslote...,#block heel shoes #loafer shoes #loafers #meta...
4,9f76d27b,HARDWARE,"[MX] 22"" Bottom Mount Drawer Slides, White Pow...",#22 #Pistola #Montaje bajo #diapositivas #Exte...
...,...,...,...,...
711223,02d80612,SHOES,[CA] Amazon Brand - find. Men’s Espadrilles,
711224,8c84db9b,SHOES,[CA] Amazon Brand - find. Men’s Espadrilles,
711225,9abc5388,SHOES,[CA] Amazon Brand - find. Men’s Espadrilles,
711226,0179aca5,SHOES,[CA] Amazon Brand - find. Men’s Espadrilles,


In [12]:
# 교집합 체크
len(set(df_image_embeddings['id'].tolist()) & set(df_merged['path'].tolist()))

398147

In [19]:
df_product = pd.merge(df_image_embeddings, df_merged, left_on='id', right_on='path', how='left').drop_duplicates(subset=['id'])
df_product = df_product[["id", 'vectors', 'product_type', 'item_name', 'item_keywords']]
df_product

,id,vectors,product_type,item_name,item_keywords
0,00000529,"{'image_embedding': [-0.03478194, -0.005159070...",HOME,[DE] UMI Slim Laundry Basket on Wheels Tall Ro...,
2,00003a93,"{'image_embedding': [-0.028530316, -0.03578147...",HOME,[ES] UMI. Essentials - Marco Doble para Sobrem...,#Marcos de Foto de estar dormitorio oficina cu...
3,000088e1,"{'image_embedding': [-0.03092741, 0.0020457155...",LAUNDRY_HAMPER,[ES] Eono Amazon Brand Doble Bolsas Cestos par...,#Gran cesta de lavanderia lavado de ropa plega...
4,0000b301,"{'image_embedding': [0.009060884, -0.030949578...",HOME_FURNITURE_AND_DECOR,[MX] Stone & Beam Alfombra Moderna a Rayas,
6,0000b9b8,"{'image_embedding': [-0.044396702, -0.01553370...",WRENCH,[ES] AmazonBasics - Juego de llaves de trinque...,
...,...,...,...,...,...
710665,ffff4040,"{'image_embedding': [-0.011574976, 0.01045859,...",CHAIR,[CA] AmazonBasics Multi-Purpose Adjustable Off...,#small #makeup #de #big #armless #medical #w/m...
710666,ffff87d7,"{'image_embedding': [-0.047705553, -0.05032951...",HUMIDIFIER,[IN] AmazonBasics Humidifier with Smart Auto-H...,#small #rooms #4010 #homech #humifiers #vicks ...
710667,ffffc378,"{'image_embedding': [0.011752412, -0.017997367...",GROCERY,"[US] 365 Everyday Value, Raspberry Geranium Fr...",#sorbet #dessert #botanical #floral #flowers #...
710668,ffffd5c6,"{'image_embedding': [-0.030390358, 0.003537298...",STORAGE_BAG,[ES] AmazonBasics 3-Bag Laundry Hamper Sorter ...,


In [14]:
# 1. 두 컬럼을 \n\n 구분자로 합쳐 새로운 item_description 컬럼 생성
df_product['item_description'] = df_product['item_name'] + '\n\n' + df_product['item_keywords']

# 2. id와 item_description 컬럼을 튜플 리스트로 변환
tuple_list = list(zip(df_product['id'], df_product['item_description']))
text_embeddings = process_parallel(tuple_list, process_text_content_embedding_worker, 64)

총 398189개의 데이터를 64개의 스레드로 처리합니다...
1000 개 처리 완료
2000 개 처리 완료
3000 개 처리 완료
4000 개 처리 완료
5000 개 처리 완료
6000 개 처리 완료
7000 개 처리 완료
8000 개 처리 완료
9000 개 처리 완료
10000 개 처리 완료
11000 개 처리 완료
12000 개 처리 완료
13000 개 처리 완료
14000 개 처리 완료
15000 개 처리 완료
16000 개 처리 완료
17000 개 처리 완료
18000 개 처리 완료
19000 개 처리 완료
20000 개 처리 완료
21000 개 처리 완료
22000 개 처리 완료
23000 개 처리 완료
24000 개 처리 완료
25000 개 처리 완료
26000 개 처리 완료
27000 개 처리 완료
28000 개 처리 완료
29000 개 처리 완료
30000 개 처리 완료
31000 개 처리 완료
32000 개 처리 완료
33000 개 처리 완료
34000 개 처리 완료
35000 개 처리 완료
36000 개 처리 완료
37000 개 처리 완료
38000 개 처리 완료
39000 개 처리 완료
40000 개 처리 완료
41000 개 처리 완료
42000 개 처리 완료
43000 개 처리 완료
44000 개 처리 완료
45000 개 처리 완료
46000 개 처리 완료
47000 개 처리 완료
48000 개 처리 완료
49000 개 처리 완료
50000 개 처리 완료
51000 개 처리 완료
52000 개 처리 완료
53000 개 처리 완료
54000 개 처리 완료
55000 개 처리 완료
56000 개 처리 완료
57000 개 처리 완료
58000 개 처리 완료
59000 개 처리 완료
60000 개 처리 완료
61000 개 처리 완료
62000 개 처리 완료
63000 개 처리 완료
64000 개 처리 완료
65000 개 처리 완료
66000 개 처리 완료
67000 개 처리 완료
68000 개 처리 완료
69000 개 처리 완료
70000 개 

In [22]:
df_text_embedding = pd.DataFrame(text_embeddings, columns=['id', 'description', 'text_embedding'])
df_golden_product = pd.merge(df_product, df_text_embedding, on='id', how='left')
# df['vectors']와 df['text_embedding']을 한 행씩 가져와 병합
df_golden_product['vectors'] = [
    {**v, 'text_embedding': t} for v, t in zip(df_golden_product['vectors'], df_golden_product['text_embedding'])
]
df_golden_product['data'] = [
    {'name': n, 'description': d} for n, d in zip(df_golden_product['item_name'], df_golden_product['item_description'])
]
df_golden_product = df_golden_product.drop(["description", "text_embedding"], axis=1)
df_golden_product

,id,vectors,product_type,item_name,item_keywords,item_description,data
0,00000529,"{'image_embedding': [-0.03478194, -0.005159070...",HOME,[DE] UMI Slim Laundry Basket on Wheels Tall Ro...,,[DE] UMI Slim Laundry Basket on Wheels Tall Ro...,{'name': '[DE] UMI Slim Laundry Basket on Whee...
1,00003a93,"{'image_embedding': [-0.028530316, -0.03578147...",HOME,[ES] UMI. Essentials - Marco Doble para Sobrem...,#Marcos de Foto de estar dormitorio oficina cu...,[ES] UMI. Essentials - Marco Doble para Sobrem...,{'name': '[ES] UMI. Essentials - Marco Doble p...
2,000088e1,"{'image_embedding': [-0.03092741, 0.0020457155...",LAUNDRY_HAMPER,[ES] Eono Amazon Brand Doble Bolsas Cestos par...,#Gran cesta de lavanderia lavado de ropa plega...,[ES] Eono Amazon Brand Doble Bolsas Cestos par...,{'name': '[ES] Eono Amazon Brand Doble Bolsas ...
3,0000b301,"{'image_embedding': [0.009060884, -0.030949578...",HOME_FURNITURE_AND_DECOR,[MX] Stone & Beam Alfombra Moderna a Rayas,,[MX] Stone & Beam Alfombra Moderna a Rayas\n\n,{'name': '[MX] Stone & Beam Alfombra Moderna a...
4,0000b9b8,"{'image_embedding': [-0.044396702, -0.01553370...",WRENCH,[ES] AmazonBasics - Juego de llaves de trinque...,,[ES] AmazonBasics - Juego de llaves de trinque...,{'name': '[ES] AmazonBasics - Juego de llaves ...
...,...,...,...,...,...,...,...
398184,ffff4040,"{'image_embedding': [-0.011574976, 0.01045859,...",CHAIR,[CA] AmazonBasics Multi-Purpose Adjustable Off...,#small #makeup #de #big #armless #medical #w/m...,[CA] AmazonBasics Multi-Purpose Adjustable Off...,{'name': '[CA] AmazonBasics Multi-Purpose Adju...
398185,ffff87d7,"{'image_embedding': [-0.047705553, -0.05032951...",HUMIDIFIER,[IN] AmazonBasics Humidifier with Smart Auto-H...,#small #rooms #4010 #homech #humifiers #vicks ...,[IN] AmazonBasics Humidifier with Smart Auto-H...,{'name': '[IN] AmazonBasics Humidifier with Sm...
398186,ffffc378,"{'image_embedding': [0.011752412, -0.017997367...",GROCERY,"[US] 365 Everyday Value, Raspberry Geranium Fr...",#sorbet #dessert #botanical #floral #flowers #...,"[US] 365 Everyday Value, Raspberry Geranium Fr...","{'name': '[US] 365 Everyday Value, Raspberry G..."
398187,ffffd5c6,"{'image_embedding': [-0.030390358, 0.003537298...",STORAGE_BAG,[ES] AmazonBasics 3-Bag Laundry Hamper Sorter ...,,[ES] AmazonBasics 3-Bag Laundry Hamper Sorter ...,{'name': '[ES] AmazonBasics 3-Bag Laundry Hamp...


In [23]:
df_golden_product.to_parquet("amazon_products_golden.parquet")

In [25]:
!gcloud storage cp amazon_products_golden.parquet gs://jk-amazon-products-index/golden-records/

uploading large objects. If you would like to opt-out and instead
perform a normal upload, run:
`gcloud config set storage/parallel_composite_upload_enabled False`
If you would like to disable this warning, run:
`gcloud config set storage/parallel_composite_upload_enabled True`
Note that with parallel composite uploads, your object might be
uploaded as a composite object
(https://cloud.google.com/storage/docs/composite-objects), which means
that any user who downloads your object will need to use crc32c
checksums to verify data integrity. gcloud storage is capable of
computing crc32c checksums, but this might pose a problem for other
clients.

Copying file://amazon_products_golden.parquet to gs://jk-amazon-products-index/golden-records/amazon_products_golden.parquet
  Completed files 32/1 | 4.8GiB/4.8GiB | 1.4GiB/s                              

Average throughput: 1.2GiB/s
